<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/869_PCFDv2_Nodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Orchestrator Nodes

### Product–Customer Fit Discovery Orchestrator v2 (PCFDv2)

This module defines the **execution nodes of the orchestrator workflow**.

Nodes represent the **control layer** of the agent — they coordinate how the system runs, but they intentionally contain very little business logic.

Instead, the heavy analytical work is delegated to specialized utility modules.

This design creates a clean separation between:

```text
Orchestration logic
and
Business intelligence logic
```

That separation is one of the key architectural strengths of the system.

---

# 🧭 Role in the Agent Architecture

The nodes form the **workflow pipeline** executed by the orchestrator.

The pipeline follows a simple and transparent structure:

```text
Goal
   ↓
Planning
   ↓
Data Loading
   ↓
Discovery
   ↓
Report Generation
```

Each node reads inputs from the agent state and writes outputs back to it.

This approach ensures the entire workflow remains:

* modular
* traceable
* easy to debug
* easy to extend

---

# 🎯 Goal Node

The `goal_node()` defines the **strategic objective of the orchestrator run**.

Rather than leaving the system direction ambiguous, the node explicitly declares the discovery mission:

```text
Discover strategic product–customer fit opportunities
```

It also identifies the intelligence domains the agent will evaluate:

```text
Customer Intelligence
Product Intelligence
Feature Intelligence
Trend Intelligence
Strategic Signals
```

This goal acts as the **anchor for the entire analysis**.

For readers of the code, this makes the orchestrator’s purpose immediately clear.

---

# 🧠 Planning Node

The `planning_node()` generates the **execution plan** for the orchestrator.

This plan is intentionally simple and deterministic.

Example structure:

```text
1. Load data
2. Run discovery
3. Generate executive report
```

Each step also defines the expected outputs.

For example:

```text
data_loading → loader_counts, snapshot timestamp
discovery → intelligence layers
report_generation → executive report
```

This step serves two important purposes.

First, it documents **how the system will operate**.

Second, it allows the orchestrator to remain **predictable and explainable**.

Many AI systems hide their workflows.
This system makes the workflow explicit.

---

# 📂 Data Loading Node

The `make_data_loading_node()` closure creates a node that loads all datasets required by the orchestrator.

Instead of embedding the loading logic directly in the node, it calls the dedicated utility:

```python
load_all_pcfd_data()
```

This keeps the node lightweight and focused on orchestration.

After loading the data, the node builds **lookup dictionaries**:

```text
customers_lookup
product_lookup
```

These structures dramatically simplify downstream operations by enabling constant-time record retrieval.

The node also returns important metadata:

```text
loader_counts
data_snapshot_loaded_at
validation_warnings
```

These signals provide transparency into the **data environment used for discovery**.

---

# 🔎 Discovery Node

The `make_discovery_node()` closure executes the discovery engine.

The node collects the required datasets from the state and passes them to the discovery utility:

```python
run_all_discovery()
```

This function performs the core analytical work across multiple intelligence layers.

Importantly, the node passes **configuration thresholds directly into the discovery engine**.

Examples include:

```text
adoption_growth_min_pct
high_margin_min_pct
segment_growth_min_pct
feature_demand_min_requests
opportunity_score_min
```

This ensures the orchestrator’s behavior is driven by **configurable business rules rather than hardcoded logic**.

Before running discovery, the node also verifies that data has been loaded.

```text
If loader_counts is missing → return error
```

This guardrail prevents the system from running analysis on incomplete state.

---

# 📄 Report Generation Node

The `make_report_node()` closure produces the final executive report.

It calls the reporting utility:

```python
build_pcfd_report()
```

This function compiles discovery outputs into a structured Markdown report.

The node then saves the report using a shared reporting tool from the toolshed:

```python
save_report()
```

This design is important for two reasons.

First, it ensures reporting behavior is **standardized across orchestrators**.

Second, it separates **report creation from report storage**, making the system easier to maintain.

The node returns two artifacts:

```text
pcfd_report → full report content
report_file_path → location of saved report
```

These outputs allow the orchestrator to integrate easily with dashboards or automated pipelines.

---

# 🧩 Why Nodes Are Intentionally Thin

A key architectural decision in this design is keeping nodes **very lightweight**.

Nodes primarily perform three tasks:

1. call the appropriate utility module
2. pass configuration values
3. write outputs to the agent state

All business logic lives in the **utilities layer**.

This separation produces several advantages.

### Easier Testing

Discovery logic can be tested independently from orchestration.

---

### Easier Reuse

The same utilities can be used by different orchestrators.

---

### Cleaner Workflows

Nodes clearly describe the system’s execution steps without being cluttered by analytical details.

---

# 🏢 Why Executives Would Value This Architecture

Most AI systems are difficult to audit because their workflows are hidden inside complex pipelines.

This design is fundamentally different.

Every stage of the process is explicit:

```text
Goal → Plan → Load Data → Discover Signals → Produce Report
```

This transparency allows leadership teams to answer critical questions such as:

* What did the system attempt to discover?
* What data was analyzed?
* What rules triggered the insights?
* What conclusions were generated?

By structuring the orchestrator this way, the system behaves less like a mysterious AI tool and more like a **structured analytical process**.

---

# ⚠ Potential Enhancements

The node architecture is already strong, but a few improvements could further increase robustness.

### Node Execution Timing

Tracking runtime for each node would allow the system to monitor pipeline performance.

Example:

```text
data_loading_time
discovery_time
report_generation_time
```

---

### Node-Level Logging

Structured logs for each node could provide better observability in production environments.

---

### Workflow Visualization

A graph representation of node dependencies could make the pipeline even easier to understand.

---

# ⭐ Final Assessment

The node layer successfully implements a **clean orchestration architecture**.

Key strengths include:

* clear workflow stages
* lightweight node design
* strong separation of orchestration and analytics
* configuration-driven execution
* reusable utility modules

This structure allows the orchestrator to behave like a **well-engineered intelligence pipeline rather than an experimental AI script**.

It demonstrates how agent systems can combine **transparent workflows, structured data processing, and configurable analytics** to produce reliable strategic insights.



In [ ]:
"""
PCFD v2 nodes: goal, planning, data_loading, discovery, report_generation.
Nodes are thin; business logic lives in orchestrator/utilities.
"""

import time
from pathlib import Path
from typing import Any, Dict

from agents.PCFDv2.orchestrator.utilities.data_loading import load_all_pcfd_data
from agents.PCFDv2.orchestrator.utilities.discovery import run_all_discovery
from agents.PCFDv2.orchestrator.utilities.report import build_pcfd_report


def goal_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Set discovery goal: find hidden opportunities in product–customer data."""
    goal = {
        "objective": "Discover strategic product–customer fit opportunities",
        "focus_areas": [
            "customer_intelligence",
            "product_intelligence",
            "feature_intelligence",
            "trend_intelligence",
            "strategic_signals",
        ],
    }
    return {
        "goal": goal,
        "errors": state.get("errors", []),
    }


def planning_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Produce execution plan (rule-based)."""
    plan = [
        {"step": 1, "name": "data_loading", "description": "Load baseline, enrichment, trends, and signals data", "outputs": ["loader_counts", "data_snapshot_loaded_at"]},
        {"step": 2, "name": "discovery", "description": "Run customer, product, feature, trend, and strategic discovery", "outputs": ["customer_intel", "product_intel", "feature_intel", "trend_intel", "strategic_signals"]},
        {"step": 3, "name": "report_generation", "description": "Build executive discovery report", "outputs": ["pcfd_report", "report_file_path"]},
    ]
    return {
        "plan": plan,
        "errors": state.get("errors", []),
    }


def make_data_loading_node(config: Any, project_root: str) -> Any:
    """Closure: data loading node with config and project_root."""

    def data_loading_node(state: Dict[str, Any]) -> Dict[str, Any]:
        errors = state.get("errors", [])
        try:
            data = load_all_pcfd_data(
                data_dir=config.data_dir,
                baseline_dir=config.baseline_dir,
                enrichment_dir=config.enrichment_dir,
                trends_dir=config.trends_dir,
                signals_dir=config.signals_dir,
                project_root=project_root,
            )
            customers = data["customers"]
            product_catalog = data["product_catalog"]
            customers_lookup = {str(c.get("Customer_ID") or c.get("customer_id")): c for c in customers if c.get("Customer_ID") or c.get("customer_id")}
            product_lookup = {str(p.get("Product_ID") or p.get("product_id")): p for p in product_catalog if p.get("Product_ID") or p.get("product_id")}
            return {
                "customers": customers,
                "product_catalog": product_catalog,
                "transactions": data["transactions"],
                "customer_metrics": data["customer_metrics"],
                "product_metrics": data["product_metrics"],
                "feature_usage": data["feature_usage"],
                "customer_journeys": data["customer_journeys"],
                "market_signals": data["market_signals"],
                "product_adoption_history": data["product_adoption_history"],
                "segment_growth_history": data["segment_growth_history"],
                "feature_demand_history": data["feature_demand_history"],
                "bundle_opportunity_signals": data["bundle_opportunity_signals"],
                "customers_lookup": customers_lookup,
                "product_lookup": product_lookup,
                "loader_counts": data["loader_counts"],
                "data_snapshot_loaded_at": data["data_snapshot_loaded_at"],
                "validation_warnings": data["validation_warnings"],
                "errors": errors,
            }
        except Exception as e:
            return {"errors": errors + [f"data_loading: {str(e)}"]}

    return data_loading_node


def make_discovery_node(config: Any) -> Any:
    """Closure: discovery node with config thresholds."""

    def discovery_node(state: Dict[str, Any]) -> Dict[str, Any]:
        errors = state.get("errors", [])
        if state.get("loader_counts") is None:
            return {"errors": errors + ["discovery: run data_loading first"]}
        data = {
            "customers": state.get("customers", []),
            "product_catalog": state.get("product_catalog", []),
            "transactions": state.get("transactions", []),
            "customer_metrics": state.get("customer_metrics", []),
            "product_metrics": state.get("product_metrics", []),
            "feature_usage": state.get("feature_usage", []),
            "market_signals": state.get("market_signals", []),
            "product_adoption_history": state.get("product_adoption_history", []),
            "segment_growth_history": state.get("segment_growth_history", []),
            "feature_demand_history": state.get("feature_demand_history", []),
            "bundle_opportunity_signals": state.get("bundle_opportunity_signals", []),
        }
        try:
            result = run_all_discovery(
                data=data,
                adoption_growth_min_pct=config.adoption_growth_min_pct,
                decline_growth_max_pct=config.decline_growth_max_pct,
                high_margin_min_pct=config.high_margin_min_pct,
                opportunity_score_min=config.opportunity_score_min,
                segment_growth_min_pct=config.segment_growth_min_pct,
                feature_demand_min_requests=config.feature_demand_min_requests,
                top_n_bundles=config.top_n_bundles,
                top_n_segments=config.top_n_segments,
                top_n_products=config.top_n_products,
            )
            return {
                "customer_intel": result["customer_intel"],
                "product_intel": result["product_intel"],
                "feature_intel": result["feature_intel"],
                "trend_intel": result["trend_intel"],
                "strategic_signals": result["strategic_signals"],
                "errors": errors,
            }
        except Exception as e:
            return {"errors": errors + [f"discovery: {str(e)}"]}

    return discovery_node


def make_report_node(config: Any, project_root: str) -> Any:
    """Closure: report node with config and project_root for reports_dir."""

    def report_node(state: Dict[str, Any]) -> Dict[str, Any]:
        errors = state.get("errors", [])
        goal = state.get("goal", {})
        loader_counts = state.get("loader_counts", {})
        data_snapshot_loaded_at = state.get("data_snapshot_loaded_at")
        validation_warnings = state.get("validation_warnings", [])
        customer_intel = state.get("customer_intel", {})
        product_intel = state.get("product_intel", {})
        feature_intel = state.get("feature_intel", {})
        trend_intel = state.get("trend_intel", {})
        strategic_signals = state.get("strategic_signals", {})
        try:
            report_md = build_pcfd_report(
                goal=goal,
                loader_counts=loader_counts,
                data_snapshot_loaded_at=data_snapshot_loaded_at,
                validation_warnings=validation_warnings,
                customer_intel=customer_intel,
                product_intel=product_intel,
                feature_intel=feature_intel,
                trend_intel=trend_intel,
                strategic_signals=strategic_signals,
            )
            from toolshed.reporting import save_report
            reports_dir = Path(project_root) / config.reports_dir if project_root else config.reports_dir
            report_path = save_report(
                report_content=report_md,
                report_id="pcfd_v2",
                reports_dir=str(reports_dir),
                prefix="pcfd_discovery",
            )
            return {
                "pcfd_report": report_md,
                "report_file_path": report_path,
                "errors": errors,
            }
        except Exception as e:
            return {"errors": errors + [f"report: {str(e)}"]}

    return report_node
